# Who worked on this project?
Muhammad Azaam Ali

We will be using dl lib pycaret which iwill help us in choosing the best model out of everyother model. This model is selected by viewing the results of operations performed on our train and val data

**Environment:** PyCaret 3.3 only runs on Python 3.11 and needs older versions of scikit-learn and pandas than the rest of the project. So this notebook runs in its own environment, `.venv-pycaret`, with the kernel **"Python 3.11 (PyCaret)"**. Setup commands are in the README.

## Load the split
We load the train and validation files that `3MushroomPredict.ipynb` saved, instead of splitting again. That way PyCaret works on exactly the same rows as our logistic regression, and the scores can be compared.

In [1]:
import pandas as pd
from pathlib import Path

# walk up from where the notebook runs to the repo root, so the data path works on every machine
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Mushroom' / 'datasets').is_dir())
DATA = ROOT / 'Mushroom' / 'datasets'

train_data = pd.read_csv(DATA / 'train.csv')
val_data = pd.read_csv(DATA / 'val.csv')

num_cols = ['cap_diameter_cm', 'stem_height_cm', 'stem_width_mm']
cat_cols = [c for c in train_data.columns if c != 'is_poisonous' and c not in num_cols]

train_data.shape, val_data.shape

((3500, 11), (750, 11))

## Setup
`setup()` tells PyCaret what the data looks like and how to test the models:
- `data=train_data`, `test_data=val_data`: train on our train set, and use our validation set as PyCaret's hold-out set instead of letting it make its own split
- `target='is_poisonous'`: the column to predict
- `categorical_features` / `numeric_features`: which columns are which, so PyCaret one-hot encodes the categories
- `normalize=True`: scales the numbers. Trees don't need it, but linear models, KNN and SVM do, and PyCaret compares all of them
- `fold=5`: every model is scored with 5-fold cross-validation on the training data, the same as our `GridSearchCV`
- `session_id=42`: same idea as `random_state=42`, so the run is repeatable
- `index=False`: our CSV files were saved without the original row numbers, so train and val both count their rows from 0. PyCaret puts them together and would see every number twice; this tells it to ignore those numbers

In [2]:
from pycaret.classification import setup, compare_models, tune_model, predict_model, pull

exp = setup(
    data=train_data,
    test_data=val_data,
    target='is_poisonous',
    categorical_features=cat_cols,
    numeric_features=num_cols,
    normalize=True,
    fold=5,
    session_id=42,
    index=False,       # train and val both number their rows from 0, so ignore those numbers
    system_log=False,  # don't write a logs.log file
    verbose=False,
)

## Compare all models
`compare_models()` trains every model type with its default settings, scores each with 5-fold cross-validation, and ranks them.

We sort on **AUC**, not on recall. Recall depends on the 0.5 threshold, and a model that calls everything poisonous gets recall 1.0 while being useless. AUC measures how well a model *ranks* poisonous above edible; a model that ranks well can always reach a high recall later by lowering the threshold.

`n_select=3` keeps the 3 best models.

In [3]:
top3 = compare_models(sort='AUC', n_select=3)
leaderboard = pull()
leaderboard

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.7931,0.8314,0.6008,0.8042,0.6871,0.5375,0.5507,0.0760
et,Extra Trees Classifier,0.7591,0.8016,0.6038,0.7156,0.6547,0.4719,0.4761,0.0760
gbc,Gradient Boosting Classifier,0.7474,0.7763,0.4453,0.7989,0.5715,0.4124,0.4479,0.1120
knn,K Neighbors Classifier,0.7400,0.7512,0.5275,0.7122,0.6058,0.4181,0.4289,0.4400
ada,Ada Boost Classifier,0.6934,0.7073,0.3766,0.6702,0.4819,0.2877,0.3120,0.0560
lr,Logistic Regression,0.6937,0.6982,0.3615,0.6807,0.4721,0.2836,0.3120,0.5780
ridge,Ridge Classifier,0.6943,0.6971,0.3457,0.6938,0.4613,0.2798,0.3130,0.0300
lda,Linear Discriminant Analysis,0.6943,0.6971,0.3517,0.6890,0.4656,0.2816,0.3130,0.0460
dt,Decision Tree Classifier,0.6923,0.6722,0.5894,0.5941,0.5914,0.3447,0.3450,0.3780
nb,Naive Bayes,0.6697,0.6657,0.1962,0.7385,0.3085,0.1799,0.2480,0.4440


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
rf,Random Forest Classifier,0.7931,0.8314,0.6008,0.8042,0.6871,0.5375,0.5507,0.076
et,Extra Trees Classifier,0.7591,0.8016,0.6038,0.7156,0.6547,0.4719,0.4761,0.076
gbc,Gradient Boosting Classifier,0.7474,0.7763,0.4453,0.7989,0.5715,0.4124,0.4479,0.112
knn,K Neighbors Classifier,0.7400,0.7512,0.5275,0.7122,0.6058,0.4181,0.4289,0.440
ada,Ada Boost Classifier,0.6934,0.7073,0.3766,0.6702,0.4819,0.2877,0.3120,0.056
lr,Logistic Regression,0.6937,0.6982,0.3615,0.6807,0.4721,0.2836,0.3120,0.578
ridge,Ridge Classifier,0.6943,0.6971,0.3457,0.6938,0.4613,0.2798,0.3130,0.030
lda,Linear Discriminant Analysis,0.6943,0.6971,0.3517,0.6890,0.4656,0.2816,0.3130,0.046
dt,Decision Tree Classifier,0.6923,0.6722,0.5894,0.5941,0.5914,0.3447,0.3450,0.378
nb,Naive Bayes,0.6697,0.6657,0.1962,0.7385,0.3085,0.1799,0.2480,0.444


### What this tells us
*(to write: which models are on top and which family do they belong to? where is logistic regression and does it match our own 0.70? single decision tree vs random forest: why the big difference?)*

We save the leaderboard so the compare notebook can show it without needing the PyCaret environment.

In [4]:
RESULTS = ROOT / 'Mushroom' / 'results'
RESULTS.mkdir(exist_ok=True)
leaderboard.to_csv(RESULTS / 'pycaret_leaderboard.csv')

## Tune the best model
`tune_model()` tries 10 random combinations of hyperparameters for the best model, again with 5-fold cross-validation, optimising AUC. `choose_better=True` keeps the original model if none of the tuned versions is better.

In [5]:
best = top3[0]
tuned = tune_model(best, optimize='AUC', n_iter=10, choose_better=True)
pull()

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7257,0.7889,0.3358,0.8476,0.4811,0.3391,0.4062
1,0.7400,0.7869,0.3736,0.8609,0.5211,0.3787,0.4409
2,0.7229,0.7423,0.3358,0.8318,0.4785,0.3333,0.3969
3,0.7214,0.7512,0.3208,0.8500,0.4658,0.3259,0.3968
4,0.7057,0.7663,0.3170,0.7706,0.4492,0.2932,0.3471
Mean,0.7231,0.7671,0.3366,0.8322,0.4791,0.3340,0.3976
Std,0.0109,0.0186,0.0200,0.0321,0.0238,0.0274,0.0300


Fitting 5 folds for each of 10 candidates, totalling 50 fits
Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
Fold,,,,,,,
0,0.7257,0.7889,0.3358,0.8476,0.4811,0.3391,0.4062
1,0.7400,0.7869,0.3736,0.8609,0.5211,0.3787,0.4409
2,0.7229,0.7423,0.3358,0.8318,0.4785,0.3333,0.3969
3,0.7214,0.7512,0.3208,0.8500,0.4658,0.3259,0.3968
4,0.7057,0.7663,0.3170,0.7706,0.4492,0.2932,0.3471
Mean,0.7231,0.7671,0.3366,0.8322,0.4791,0.3340,0.3976
Std,0.0109,0.0186,0.0200,0.0321,0.0238,0.0274,0.0300


## Score on the validation set
`predict_model()` without new data scores the model on the hold-out set we gave to `setup()`: our validation set. These numbers can be compared directly with the logistic regression scores in `3MushroomPredict.ipynb`.

In [6]:
predict_model(tuned)
pull()

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.7920,0.8377,0.6092,0.7936,0.6892,0.5370,0.5476


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,Random Forest Classifier,0.792,0.8377,0.6092,0.7936,0.6892,0.537,0.5476


### What this tells us
*(to write: did tuning help? is the validation AUC close to the cross-validation AUC? which models will we tune ourselves next, and which ones do we drop, and why?)*